In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries

from darts.utils.missing_values import missing_values_ratio, fill_missing_values
from darts.metrics.metrics import mape, mae, rmse, mase

from ts_missing_values.gap_filling import *
from ts_missing_values.comparison import *
from ts_missing_values.preprocessing import *
from ts_missing_values.utility import *

from tqdm import tqdm

In [ ]:
sensors = pd.read_csv('./data/synthetic_traffic.csv')
sensors.timestamp = pd.to_datetime(sensors.timestamp)
sensors = sensors.set_index('timestamp')

Wrap each sensor into a Darts TimeSeries, plot it for inspection, and store series and names into a dict and list for future use.

In [ ]:
sensors_dict = {}

fig, ax = plt.subplots(10,3, figsize=(15,10))

for i in range(10):
    for j in range(3):
        name = sensors.columns[j+i*3]
        ts_original = TimeSeries(times=sensors.index, values=sensors[name].values)
        ts_original.plot(ax=ax[i,j])
        ax[i,j].get_legend().remove()
        sensors_dict[name] = ts_original
        
sensors_list = list(sensors_dict.values())
sensors_names = list(sensors_dict.keys())

## linear interpolation validation

In [ ]:
def print_metrics(name, metrics):
    metric_for_each_series = np.nanmean(np.array(metrics), axis=1)
    
    print(f'matrics for {name}')
    print('====================')
    print(f'min {np.nanmin(metric_for_each_series):.2f}')
    print(f'25Q {np.quantile(metric_for_each_series, 0.25):.2f}')
    print(f'mean {np.mean(metric_for_each_series):.2f}')
    print(f'median {np.median(metric_for_each_series):.2f}')
    print(f'75Q {np.quantile(metric_for_each_series, 0.75):.2f}')
    print(f'max {np.nanmax(metric_for_each_series):.2f}')

### Gaps

In [ ]:
linear_MAE_results = []
linear_RMSE_results = []

for i in tqdm(range(len(sensors_list))):
    series = sensors_list[i]
    mae_results = []
    rmse_results = []

    for i in range(50):
        series_with_gap, artificial_gap = create_artificial_gap(series, plot=False, return_removed_slice=True)
        auto_filled_series = fill_missing_values(series_with_gap)

        a = series.slice_intersect(artificial_gap)
        b = auto_filled_series.slice_intersect(artificial_gap)

        # vals = a.univariate_values()
        # print ((~np.isnan(vals)).sum())

        mae_results.append(mae(a, b))
        rmse_results.append(rmse(a,b))

    linear_MAE_results.append(mae_results)
    linear_RMSE_results.append(rmse_results)

In [ ]:
print_metrics('MAE', linear_MAE_results)

In [ ]:
print_metrics('RMSE', linear_RMSE_results)

### Sporadic missing values

In [ ]:
series = sensors_list[1]
s, ov = create_artificial_sporadic_missing_values(series, 10, True)
ov